# SubEvent2Hap · Colab 생성 서버

**런타임 → 런타임 유형 변경 → GPU(A100 또는 L4)** 를 고른 뒤 **런타임 → 모두 실행** 하세요.

마지막 셀에 나오는 주소를 에디터나 Story 페이지 상단의 **Colab** 칸을 눌러 붙여 넣고, 생성 모델에서 **(Colab GPU)** 를 고르면 됩니다.
이 노트북이 켜져 있는 동안만 연결됩니다 (창을 닫거나 런타임이 끊기면 다시 모두 실행).

In [ ]:
# 1. 코드, 패키지, HapticGen 가중치 (처음 3~5분)
!nvidia-smi -L
import os
if not os.path.exists('/content/SubEvent2Hap'):
    !git clone -q https://github.com/jaehyeonghwang/SubEvent2Hap.git /content/SubEvent2Hap
%cd /content/SubEvent2Hap
!git pull -q
!git submodule update -q --init HapticGen
!pip -q install -e HapticGen --no-deps
!pip -q install -r requirements-colab.txt
from huggingface_hub import snapshot_download
snapshot_download('HapticGen/HapticGen-Weights', local_dir='weights/hapticgen')
print('준비 완료')

In [ ]:
# 2. 생성 서버 + 공개 터널 (cloudflared, 계정 불필요)
import secrets, subprocess, re, time, threading, requests
token = secrets.token_urlsafe(16)
worker = subprocess.Popen(['python', 'scripts/colab_worker.py', '--port', '8000'],
                          env=dict(os.environ, WORKER_TOKEN=token, PYTHONWARNINGS='ignore'),
                          stdout=open('/content/worker.log', 'w'), stderr=subprocess.STDOUT)
if not os.path.exists('/content/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        public = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()   # keep draining the pipe
for _ in range(120):
    try:
        if requests.get('http://127.0.0.1:8000/health', headers={'X-Token': token}, timeout=2).ok:
            break
    except Exception:
        time.sleep(2)
print('HapticGen 모델 불러오는 중...')
print(requests.get('http://127.0.0.1:8000/info', params={'model_name': 'hapticgen'}, headers={'X-Token': token}, timeout=900).json())

In [ ]:
# 3. 이 주소를 복사해서 웹 페이지의 Colab 칸에 붙여 넣으세요
print(f'{public}?token={token}')

In [ ]:
# 4. (선택) 켜 두는 동안 상태 확인. 생성 요청이 들어오면 로그가 늘어납니다.
while True:
    alive = worker.poll() is None and tunnel.poll() is None
    print(time.strftime('%H:%M:%S'), '정상' if alive else '중단됨: 위 셀들을 다시 실행하세요', flush=True)
    if not alive:
        print(open('/content/worker.log').read()[-2000:])
        break
    time.sleep(300)